In [0]:
%python
# Databricks notebook source
# DBTITLE 1, Lendo a tabela agregada da Camada Gold
df_summary = spark.read.table("credit_risk.gold.gold_loan_summary_by_purpose")

# COMMAND ----------
# DBTITLE 2, Transformando dados tabulares em texto descritivo (Chunks para RAG)
from pyspark.sql.functions import col, concat, lit

df_text_summary = df_summary.select(
    concat(
        lit("O propósito do empréstimo é "), col("purpose"), 
        lit(". O total de empréstimos é de "), col("total_emprestimos"),
        lit(" com valor médio de crédito de "), col("valor_medio_credito"),
        lit(" e prazo médio de "), col("prazo_medio_meses"), lit(" meses.")
    ).alias("chunk_text"),
    col("purpose").alias("id_key")
)

# COMMAND ----------
# DBTITLE 3, Salvando na Gold Textual (Delta Lake)
df_text_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("credit_risk.gold.gold_credit_text_summary")

print("Tabela Gold Textual populada com sucesso para o RAG!")